<a href="https://colab.research.google.com/github/cameronspeirs2-byte/freight-rate-dynamics-dissertation-code/blob/main/AIS_variable_creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 1: setup & helper functions

from google.colab import drive
drive.mount('/content/drive')

import os, re, zipfile, tempfile
import pandas as pd
import numpy as np
from tqdm import tqdm

# Paths
ZIP_DIR = "/content/drive/MyDrive/AIS_Project/raw_zips"
DAILY_CSV = "/content/drive/MyDrive/AIS_Project/daily_data.csv"

# West Coast bounding box
WC_lat_min, WC_lat_max = 30.0, 50.0
WC_lon_min, WC_lon_max = -130.0, -115.0

# Port boxes
PORT_BOXES = {
    "LosAngeles": {"lat_min": 33.687, "lat_max": 33.777, "lon_min": -118.322, "lon_max": -118.212},
    "LongBeach":  {"lat_min": 33.714, "lat_max": 33.794, "lon_min": -118.260, "lon_max": -118.170},
    "Oakland":    {"lat_min": 37.760, "lat_max": 37.830, "lon_min": -122.355, "lon_max": -122.285},
    "Seattle":    {"lat_min": 47.564, "lat_max": 47.634, "lon_min": -122.376, "lon_max": -122.306},
    "Tacoma":     {"lat_min": 47.225, "lat_max": 47.305, "lon_min": -122.470, "lon_max": -122.390},
}

# --- extract date from filename AIS_2017_04_06.zip ---
def extract_date(fname):
    m = re.search(r"(20\d{2})_(\d{2})_(\d{2})", fname)
    if not m:
        return None
    y, mth, d = map(int, m.groups())
    return pd.Timestamp(y, mth, d)

# --- read a CSV from a ZIP using pandas (tolerant of bad rows) ---
def read_csv_from_zip(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        csv_name = [f for f in z.namelist() if f.endswith(".csv")][0]

        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".csv")
        tmp.write(z.read(csv_name))
        tmp.close()

        df = pd.read_csv(
            tmp.name,
            engine="python",
            on_bad_lines="skip"
        )

        # Ensure required cols exist
        required = ["MMSI", "BaseDateTime", "LAT", "LON", "SOG"]
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise ValueError(f"Missing required columns {missing} in {zip_path}")

        df["BaseDateTime"] = pd.to_datetime(df["BaseDateTime"], errors="coerce")
        df = df.dropna(subset=["BaseDateTime"])

        return df

# --- process one day's ZIP into daily metrics ---
def process_one_day(day, path):
    try:
        df = read_csv_from_zip(path)
    except Exception as e:
        print(f"FAILED TO READ {path}: {e}")
        return None

    # Filter West Coast region
    df_wc = df[
        (df["LAT"] >= WC_lat_min) &
        (df["LAT"] <= WC_lat_max) &
        (df["LON"] >= WC_lon_min) &
        (df["LON"] <= WC_lon_max)
    ]

    total_records = len(df_wc)
    slow_records = len(df_wc[df_wc["SOG"] < 1])
    active_ships = df_wc["MMSI"].nunique()

    # Port arrivals + turnaround
    port_arrivals = {p: set() for p in PORT_BOXES}
    turnaround_first = {}
    turnaround_last = {}

    for port, box in PORT_BOXES.items():
        df_port = df[
            (df["LAT"] >= box["lat_min"]) &
            (df["LAT"] <= box["lat_max"]) &
            (df["LON"] >= box["lon_min"]) &
            (df["LON"] <= box["lon_max"])
        ]

        if len(df_port) == 0:
            continue

        port_arrivals[port] = set(df_port["MMSI"].unique())

        for mmsi, group in df_port.groupby("MMSI"):
            tmin = group["BaseDateTime"].min()
            tmax = group["BaseDateTime"].max()
            key = (port, mmsi)

            if key not in turnaround_first:
                turnaround_first[key] = tmin
                turnaround_last[key] = tmax
            else:
                turnaround_first[key] = min(turnaround_first[key], tmin)
                turnaround_last[key] = max(turnaround_last[key], tmax)

    durations = []
    for key, t1 in turnaround_first.items():
        t2 = turnaround_last[key]
        hrs = (t2 - t1).total_seconds() / 3600
        if hrs >= 0:
            durations.append(hrs)

    avg_turnaround = np.mean(durations) if durations else np.nan
    total_port_arrivals = len(set().union(*port_arrivals.values())) if port_arrivals else 0

    row = {
        "date": str(day.date()),
        "daily_active_ships": active_ships,
        "daily_delay_rate": slow_records / total_records if total_records > 0 else np.nan,
        "daily_port_arrivals": total_port_arrivals,
        "daily_avg_turnaround_hours": avg_turnaround,
    }

    for p in PORT_BOXES:
        row[f"arrivals_{p}"] = len(port_arrivals[p])

    return row

print("Setup complete ✅")

Mounted at /content/drive
Setup complete ✅


In [ ]:
# ----------------------------------------------------
# SECTION 3: PROCESS ALL DAYS WITH RESUME LOGIC
# ----------------------------------------------------

# 1. Build sorted list of (date, path) for all ZIP files
files = []
for f in os.listdir(ZIP_DIR):
    if f.endswith(".zip"):
        dt = extract_date(f)
        if dt is not None:
            files.append((dt, os.path.join(ZIP_DIR, f)))

files = sorted(files, key=lambda x: x[0])
print("Total ZIPs detected:", len(files))


# 2. Read already completed days from daily_data.csv
if os.path.exists(DAILY_CSV):
    done_days = set(pd.read_csv(DAILY_CSV)["date"].astype(str))
else:
    done_days = set()

print("Days already processed:", len(done_days))


# 3. Create header if file doesn't exist
if not os.path.exists(DAILY_CSV):
    cols = [
        "date",
        "daily_active_ships",
        "daily_delay_rate",
        "daily_port_arrivals",
        "daily_avg_turnaround_hours",
    ] + [f"arrivals_{p}" for p in PORT_BOXES]

    pd.DataFrame(columns=cols).to_csv(DAILY_CSV, index=False)
    print("Created new daily_data.csv file with header.")


# 4. MAIN LOOP — process only days NOT already in the CSV
rows_written = 0

for day, path in tqdm(files):

    day_str = str(day.date())

    # Skip if already done
    if day_str in done_days:
        continue

    # Process the day
    row = process_one_day(day, path)

    # If successfully processed, append to CSV
    if row is not None:
        pd.DataFrame([row]).to_csv(
            DAILY_CSV,
            mode="a",
            header=False,
            index=False
        )
        rows_written += 1
        done_days.add(day_str)  # Update the resume set

print("--------------------------------------------------")
print("New days written in THIS run:", rows_written)

# 5. Show the updated dataset info
df_daily = pd.read_csv(DAILY_CSV)
print("Total rows in daily_data.csv now:", df_daily.shape[0])
print("Last few rows:")
print(df_daily.tail())

Total ZIPs detected: 1095
Days already processed: 831


100%|██████████| 1095/1095 [7:35:19<00:00, 24.95s/it]

--------------------------------------------------
New days written in THIS run: 264
Total rows in daily_data.csv now: 1119
Last few rows:
            date  daily_active_ships  daily_delay_rate  daily_port_arrivals  \
1114  2019-12-27                3306          0.846480                  439   
1115  2019-12-28                3400          0.822996                  415   
1116  2019-12-29                3360          0.803059                  433   
1117  2019-12-30                3329          0.830542                  436   
1118  2019-12-31                3326          0.830117                  425   

      daily_avg_turnaround_hours  arrivals_LosAngeles  arrivals_LongBeach  \
1114                   16.938154                  178                 159   
1115                   16.533182                  168                 151   
1116                   15.704875                  180                 157   
1117                   15.828355                  172                 156   
1